# EIM–CMB — run everything (Colab)

**No local Python required.** This notebook clones the repo, installs dependencies, and runs the
whole pipeline end to end: data-independent self-tests → reproduce the paper's §III numbers →
the §V zonal signature + Figure 4 → Figures 1–3 → **download the real Planck 2018 maps** →
full closure run over four maps × two masks.

**To run:** Runtime → **Run all**. First run takes ~20–40 min (the Planck download dominates).
Colab sessions are ephemeral — results are regenerated each time; the last cell shows how to save
them to Google Drive.

In [ ]:
#@title Setup — clone repo + install (edit GITHUB_USER once)
GITHUB_USER = "skankcunt42"   #@param {type:"string"}
REPO = "eim-cmb"
import os, sys
if os.path.basename(os.getcwd()) != REPO:
    if not os.path.isdir(REPO):
        !git clone --depth 1 https://github.com/{GITHUB_USER}/{REPO}.git
    os.chdir(REPO)
print("cwd:", os.getcwd())
!pip -q install healpy camb astropy numpy scipy matplotlib nbformat
sys.path.insert(0, os.path.abspath("engine"))
import importlib
print("environment ready")

In [ ]:
#@title Environment check
import healpy, numpy, scipy, matplotlib
try:
    import camb; cv = camb.__version__
except Exception as e:
    cv = f"MISSING ({e}) — CAMB optional; a ΛCDM-plateau fallback is used"
print("healpy", healpy.__version__, "| numpy", numpy.__version__, "| camb", cv)

## Part 1 — data-independent self-tests
Theorem 1 (icosahedral selection), Theorem 2 (k-fold azimuthal selection), and the sectoral/zonal
estimator self-tests. These are deterministic: they either reproduce the paper's machine-precision
facts or they don't. No data needed.

In [ ]:
import selection_rules
selection_rules.main()

## Part 2 — reproduce the paper's §III numbers
Independent regeneration of Table I / §III from the engine (matched Monte Carlo). Raise `N` to
tighten the error bars.

In [ ]:
import reproduce_paper_numbers as R
R.main(400)

## Part 3 — the §V zonal signature + Figure 4
The observable the hypothesis actually predicts: zonal axis estimator + (zonal_low, m5_mid) +
matched null. Genuine template → (1,1); sectoral sky and null → the low corner.

In [ ]:
import zonal_signature as Z
Z.main()
from IPython.display import Image, display
display(Image("engine/zonal_signature_fig4.png"))

## Part 4 — Figures 1–3 (parity, co-axiality, ledger)

In [ ]:
import make_figures
make_figures.main(400)
from IPython.display import Image, display
for f in ["engine/fig1_parity.png","engine/fig2_coaxiality.png","engine/fig3_ledger.png"]:
    display(Image(f))

## Part 5 — fetch the REAL Planck 2018 maps
Colab has open internet, so the streaming fetch that a locked sandbox can't do runs here:
downloads the four component-separated maps + two real masks, keeps only temperature, downgrades to
Nside=64, and deletes each ~1 GB raw file immediately (peak disk ≈ one raw map).

*Tip:* for a fast first pass, uncomment the trim line to fetch just SMICA+NILC (~2 maps).

In [ ]:
import stream_fetch as F
# --- quick smoke test (2 maps): uncomment the next line ---
# F.MAPS = {k: F.MAPS[k] for k in ["smica","nilc"]}
ok = F.preflight()
if ok:
    F.run_fetch()
    F.verify()
else:
    print("Preflight failed — check the messages above (network/disk).")

## Part 6 — full closure run on the real maps
Freezes the axes, builds a matched null through each mask, and reports both the legacy anomaly set
{g, α, H₅} and the §V framework test {zonal_low, m5_mid} per map × mask, with the two decision
rules. If the maps from Part 5 are present it runs on the **real data**; otherwise it falls back to
a labelled ΛCDM dry-run.

In [ ]:
import importlib, run_closure
importlib.reload(run_closure)
run_closure.main(300)

## Save results to Google Drive (optional)
Colab is ephemeral; run this to copy the figures and downgraded maps to your Drive.

In [ ]:
# from google.colab import drive; drive.mount("/content/drive")
# import os, glob, shutil
# out = "/content/drive/MyDrive/eim-cmb-results"; os.makedirs(out, exist_ok=True)
# for f in glob.glob("engine/*.png") + glob.glob("*_nside64_uK.fits") + glob.glob("*.npz"):
#     shutil.copy(f, out)
# print("saved to", out)

---
**Next:** the numbers from Part 6 populate Tables II–III of `paper/EIM_CMB_closure_paper_FINAL.docx`
(§VI). Per the group-theoretic obstruction (§IV) the expected framework result is negative — the
data land in the null cloud of Figure 4. Commit code/doc changes back through GitHub; leave the
regenerated `.fits`/`.png` out of version control (see `.gitignore`).